## Write Urself Challenge
this is a section where I have to rewrite everything from memory and figure it out myself

In [61]:
import math
import random

In [62]:
class Value:
    def __init__(self, num, _child=(), label=''):
        self.data   = num
        self.label  = label

        self._prev  = set(_child)
        self._backward = lambda: None
        self.grad   = 0.0

    # print
    def __repr__(self):
        return f"Value({self.label}:{self.data})"


    # basic operations
    def __add__ (self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad  += other.data * out.grad
            other.grad +=  self.data * out.grad 
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1 

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def exp(self):
        out = Value(math.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    # activation func
    def sigmoid(self):
        out = Value(1 / (1 + (-self).exp().data), (self,))

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        t = max(0, self.data)
        out = Value(t, (self,))
        
        def _backward():
            self.grad += (1 if t > 0 else 0) * out.grad
        out._backward = _backward

        return out

    # backprop
    def backward(self):
        topo=[]
        vis =set()

        def build_topo(val):
            if val not in vis:
                vis.add(val)
                for c in val._prev:     
                    build_topo(c)
                topo.append(val)

        build_topo(self)
        self.grad = 1.0
        for val in reversed(topo):
            val._backward()

In [59]:
a = Value(4.0, label='a');
b = Value(3.0, label='b')
c = Value(1.0, label='c')

d = a * b; d.label='d'
e = d + c; e.label='e'
s = e.relu()

s.backward()
s


Value(:13.0)

In [60]:
print(s.grad)
print(e.grad)
print(d.grad)
print(c.grad)
print(b.grad)
print(a.grad)

1.0
1.0
1.0
1.0
4.0
3.0


### NEURAL NETWORK I AM COMING BABYYYY

In [ ]:
class Neuron:
    def __init__(self, nin, activation=None):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b =  Value(random.uniform(-1,1))

        acts = {
            'relu': lambda x: x.relu(),
            'sigmoid': lambda x: x.sigmoid(),
            'tanh': lambda x: x.tanh(),
            'identity': lambda x: x,
        }
        self.activation = acts[activation]

    def __call__(self, x):
        out = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        return self.activation(out)

    def parameters(self):
        return [self.b] + self.w

In [106]:
class Layer:
    def __init__(self, nin, nout, activation=None):
        self.neurons = [Neuron(nin, activation=activation) for _ in range(nout)] 

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [462]:
class MLP:
    def __init__(self, nin, nouts):
        self.layers = []
        prev = nin

        for nout, act in nouts:
            self.layers.append(Layer(nin, nout, activation=act))
            prev = nout

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self, X, y, epochs, lr=0.1, batch_size=None, mode='sgd', debug=False, iter=50):
        def make_batches(X, y, batch_size, mode):
            idx = list(range(len(X)))

            if batch_size is not None and mode == 'sgd':
                random.shuffle(idx)

            if batch_size is None:
                yield X, y
                return

            for i in range(0, len(idx), batch_size):
                batch_idx = idx[i:i + batch_size]
                xb = [X[j] for j in batch_idx]
                yb = [y[j] for j in batch_idx]
                yield xb, yb

        for epoch in range(epochs):
            for xb, yb in make_batches(X, y, batch_size=batch_size, mode=mode):
                # reset grads
                for p in n.parameters():
                    p.grad = 0.0

                # forward
                ypred = [self(x) for x in xb]

                # loss
                loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, yb))

                # backward
                loss.backward()

                # update
                for p in n.parameters():
                    p.data += -lr * p.grad

            # debug
            if debug and epoch % iter == 0:
                print(f'Epoch {epoch}: loss = {loss.data}')


    def predict(self, X):
        return [self(x) for x in X]

In [463]:
# 8 samples with 4 inputs each
xs = [
    [ 1.5,  2.0,  1.0,  0.5], # (1.5*2.0) - (1.0^2) + 0.5 = 3.0 - 1.0 + 0.5 = +2.5 -> 1.0
    [ 2.0, -1.0,  1.5, -0.5], # (2.0*-1.0) - (1.5^2) - 0.5 = -2.0 - 2.25 - 0.5 = -4.75 -> 0.0
    [-1.0, -2.0,  1.0, -0.5], # (-1.0*-2.0) - (1.0^2) - 0.5 = 2.0 - 1.0 - 0.5 = +0.5 -> 1.0
    [ 0.5,  1.0,  2.0,  1.0], # (0.5*1.0) - (2.0^2) + 1.0 = 0.5 - 4.0 + 1.0 = -2.5 -> 0.0
    [ 2.0,  2.0,  0.5, -1.0], # (2.0*2.0) - (0.5^2) - 1.0 = 4.0 - 0.25 - 1.0 = +2.75 -> 1.0
    [-1.5,  1.0,  1.0,  0.0], # (-1.5*1.0) - (1.0^2) + 0.0 = -1.5 - 1.0 + 0.0 = -2.5 -> 0.0
    [ 1.0,  3.0,  1.2, -0.2], # (1.0*3.0) - (1.2^2) - 0.2 = 3.0 - 1.44 - 0.2 = +1.36 -> 1.0
    [-2.0,  0.5,  1.5, -1.0], # (-2.0*0.5) - (1.5^2) - 1.0 = -1.0 - 2.25 - 1.0 = -4.25 -> 0.0
]

ys = [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]

In [466]:
# 4 inputs, 2 hidden layers with ReLU, 1 output neuron with Sigmoid
n = MLP(4, [
    (6, 'relu'),
    (6, 'relu'),
    (1, 'sigmoid')
])

n.train(xs, ys, 1000, batch_size=4, debug=True)

Epoch 0: loss = 0.7379648802222416
Epoch 50: loss = 0.003746564167734055
Epoch 100: loss = 0.000988726762644982
Epoch 150: loss = 0.0009820577245289643
Epoch 200: loss = 0.00037153756521468464
Epoch 250: loss = 0.0005569758650094148
Epoch 300: loss = 0.00035821933233753554
Epoch 350: loss = 0.0003376052560318631
Epoch 400: loss = 0.000173116285491629
Epoch 450: loss = 0.00030732396918201993
Epoch 500: loss = 0.00019501093821212406
Epoch 550: loss = 0.00025044596562783764
Epoch 600: loss = 0.0002491185632243524
Epoch 650: loss = 9.605604680246094e-05
Epoch 700: loss = 0.00010235359739020244
Epoch 750: loss = 0.0001950505563002722
Epoch 800: loss = 7.647255612523157e-05
Epoch 850: loss = 0.00011948787419590137
Epoch 900: loss = 0.00013414561136612196
Epoch 950: loss = 0.00011387698437270393


In [ ]:
# sgd -> Epoch 950: loss = 0.0005600735717141948
# all -> Epoch 950: loss = 0.00024534024758760903
# batch-4 -> Epoch 550: loss = 0.00025044596562783764
# Epoch 600: loss = 0.0002491185632243524
# Epoch 650: loss = 9.605604680246094e-05
# Epoch 700: loss = 0.00010235359739020244
# Epoch 750: loss = 0.0001950505563002722
# Epoch 800: loss = 7.647255612523157e-05
# Epoch 850: loss = 0.00011948787419590137
# Epoch 900: loss = 0.00013414561136612196
# Epoch 950: loss = 0.00011387698437270393
x_test = [[1.8, 1.5, 1.1, -0.5]]

n.predict(x_test)

[Value(:0.8098164277951255)]